# Ingest qualifying folder assingment


## Assignment
- create a DF loading the qualifying folder in the raw container
- rename columns to (quialyfing_id,race_id,driver_id,constructor_id)
- add a new columns ingestion_timestampt
- save the file in parquet file in the processed container.
- verify the schema of the partquet file

In [0]:
%run "../includes/common_functions"

In [0]:
%run "../includes/configuration"

In [0]:
dbutils.widgets.text("p_date_source","")
#dbutils.widgets.dropdown("p_date_source","Testing",["Testing","Production"])
v_data_source=dbutils.widgets.get("p_date_source")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType,DateType
from pyspark.sql.functions import current_timestamp,to_date,current_date,lit,to_timestamp,concat,col


In [0]:
qualifying_schema = StructType(fields=[StructField("qualifyId", IntegerType(), False),StructField("raceId", IntegerType(), True),StructField("driverId", IntegerType(), True),StructField("constructorId", IntegerType(), True),StructField("number", IntegerType(), True),StructField("position", IntegerType(), True),StructField("q1", StringType(), True),StructField("q2", StringType(), True),StructField("q3", StringType(), True)])
qualifying_df = spark.read.schema(qualifying_schema).option("multiline",True).json(f"{raw_folder_path}/lap_times")

In [0]:
qualifying_df = qualifying_df.withColumnRenamed('raceId','race_id').withColumnRenamed('driverId','driver_id').withColumnRenamed('constructorId','constructor_id').withColumnRenamed('qualifyingId','qualifying_id')

In [0]:
quialifying_df=add_ingestion_timestamp(qualifying_df)
quialifying_df = add_data_source(quialifying_df,v_data_source)

In [0]:
#display(qualifying_df)
#qualifying_df.count()

## Write DF into parquet file

In [0]:
qualifying_df.write.mode("overwrite").partitionBy("race_id").parquet(f"{processed_folder_path}/qualifying")

In [0]:
#df=spark.read.parquet(f"{processed_folder_path}/qualifying")
#df.printSchema()

In [0]:
dbutils.notebook.exit("Success")